# 01 - Exploring games.csv

Goal: understand what the Steam dataset (FronkonGames, Kaggle) contains before cleaning it.

In [1]:
import sys
sys.path.append('..')   # lets the notebook import the src/ folder from notebooks/

import pandas as pd
from src.data_loader import load_games

DATA_PATH = '../data/games.csv'

## 1. The trap: a one-column shift

We first load the file "naively", on 3 rows only.

In [2]:
naive = pd.read_csv(DATA_PATH, nrows=3)
naive[['Name', 'Release date', 'Price', 'Tags']]

,Name,Release date,Price,Tags
2539430,"Aug 1, 2023",0 - 0,0,NaN
496350,"Jul 29, 2016",0 - 20000,65,"Adventure,Visual Novel,Anime,Cute"
1034400,"May 6, 2019",0 - 20000,0,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."


The `Name` column holds a date, and `Price` is 65 for a $5.24 game: everything is shifted by one column.
We count the names in the header and the values in a row:

In [3]:
with open(DATA_PATH, encoding='utf-8') as f:
    header = f.readline()
print('Names in the header:', len(header.split(',')))
print('Values in a row:', len(naive.columns) + 1, '(columns + index)')
for name in header.split(','):
    if 'DLC' in name:
        print('Glued name:', name)

Names in the header: 39
Values in a row: 40 (columns + index)
Glued name: DiscountDLC count


**Explanation**: the header has 39 names but each row has 40 values, because `Discount` and `DLC count` are glued together as `DiscountDLC count`.
pandas then uses the extra value (AppID) as the index, and each name ends up above the next column.
**Fix**: `load_games` (in `src/data_loader.py`) skips the header and passes the list of the 40 correct names with the `names` parameter of `pd.read_csv`.

## 2. Fixed loading (slow cell: run it only once)

In [4]:
USEFUL_COLUMNS = ['AppID', 'Name', 'Release date', 'Estimated owners',
                  'Price', 'Positive', 'Negative', 'Tags']
df = load_games(DATA_PATH, usecols=USEFUL_COLUMNS)
print(df.shape)
df.columns

(125855, 8)


Index(['AppID', 'Name', 'Release date', 'Estimated owners', 'Price',
       'Positive', 'Negative', 'Tags'],
      dtype='str')

In [5]:
df[['Name', 'Price', 'Positive', 'Negative', 'Tags']].head()

,Name,Price,Positive,Negative,Tags
0,Black Dragon Mage Playtest,0.00,0,0,NaN
1,Supipara - Chapter 1 Spring Has Come!,5.24,252,3,"Adventure,Visual Novel,Anime,Cute"
2,Mystery Solitaire The Black Raven,4.99,21,3,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."
3,버튜버 파라노이아 - Vtuber Paranoia,8.99,0,0,NaN
4,Maze Quest VR,4.99,0,0,NaN


The check is good: a game name, a price in dollars, two review counts and tags. About 125,000 rows.

## 3. Types and missing values

In [6]:
df.dtypes

AppID                 int64
Name                    str
Release date            str
Estimated owners        str
Price               float64
Positive              int64
Negative              int64
Tags                    str
dtype: object

In [7]:
df.isna().sum()

AppID                   0
Name                    1
Release date            0
Estimated owners        0
Price                   0
Positive                0
Negative                0
Tags                42502
dtype: int64

The numbers (`Price`, `Positive`, `Negative`) are already numeric. The date and the owners are text: they will need to be converted during cleaning.
About one game in three has no tag: these games cannot be used as comparable games.

## 4. How many free games?

In [8]:
n_free = (df['Price'] == 0).sum()
print(n_free, 'free games, i.e.', round(100 * n_free / len(df), 1), '%')

26661 free games, i.e. 21.2 %


About 21 % of the games are free. Without a price, no sales revenue can be estimated: they will be removed.

## 5. What is the price distribution?

In [9]:
paid = df[df['Price'] > 0]
paid['Price'].describe()

count    99194.000000
mean         6.104304
std         13.784581
min          0.490000
25%          1.490000
50%          3.590000
75%          6.990000
max        999.980000
Name: Price, dtype: float64

In [10]:
price_bins = [0, 5, 10, 15, 20, 30, 1000]
pd.cut(paid['Price'], bins=price_bins).value_counts().sort_index()

Price
(0, 5]        66830
(5, 10]       20070
(10, 15]       6742
(15, 20]       2977
(20, 30]       1431
(30, 1000]     1144
Name: count, dtype: int64

The vast majority of paid games cost less than $10. The median price of a paid game is low: the market is dominated by small games.

## 6. How many games have fewer than 10 reviews?

In [11]:
n_reviews = df['Positive'] + df['Negative']
few = (n_reviews < 10).sum()
print(few, 'games have fewer than 10 reviews, i.e.', round(100 * few / len(df), 1), '%')

69193 games have fewer than 10 reviews, i.e. 55.0 %


**Striking figure**: more than half of Steam games (55 %) have fewer than 10 reviews. Most games go almost unnoticed.

## 7. How many games are released per year?

In [12]:
# errors='coerce': an unreadable date becomes an empty cell (NaT) instead of crashing.
release = pd.to_datetime(df['Release date'], format='%b %d, %Y', errors='coerce')
print('Unreadable dates:', release.isna().sum())
release.dt.year.value_counts().sort_index().tail(12)

Unreadable dates: 0


Release date
2015     2515
2016     4141
2017     5922
2018     7461
2019     7243
2020     8804
2021    11067
2022    12288
2023    14598
2024    20033
2025    24973
2026     3316
Name: count, dtype: int64

Releases explode: about 7,000 games a year around 2018, and nearly 25,000 in 2025. Competition grows very fast. The year 2026 is incomplete (the dataset stops before the end of the year).

## 8. What are the 20 most frequent tags?

In [13]:
# A game's tags are in a single string: split turns it into a list,
# then explode creates one row per tag, which can finally be counted.
tags = df['Tags'].str.split(',').explode()
tags.value_counts().head(20)

Tags
Singleplayer      50353
Indie             48556
Action            36812
Casual            36565
Adventure         35200
2D                26794
3D                17997
Simulation        17896
Strategy          17850
Puzzle            16370
Atmospheric       16284
RPG               15745
Colorful          14261
Pixel Graphics    13869
Story Rich        13345
Exploration       13241
Cute              12680
First-Person      12295
Fantasy           11501
Arcade            11495
Name: count, dtype: int64

`Indie` is the 2nd most frequent tag (almost 50,000 games): the indie segment is huge and very competitive. Genre tags (Action, Casual, Adventure) dominate; more specific niches (Roguelike, Deckbuilding...) are studied in notebook 04.

## 9. Owners (SteamSpy estimate)

In [14]:
df['Estimated owners'].value_counts().head(8)

Estimated owners
0 - 20000            75705
0 - 0                24579
20000 - 50000        11397
50000 - 100000        5355
100000 - 200000       3454
200000 - 500000       2856
500000 - 1000000      1155
1000000 - 2000000      729
Name: count, dtype: int64

Owners are given as text ranges ("20000 - 50000"). 80 % of the games are in "0 - 20000" or "0 - 0": the range is too wide to estimate sales, hence the review-based method (notebook 03).